# 32×32 LED Matrix Preview

Preview the pixel composition of `base.py`, `halloween.py`, or `birthday.py` without connecting to the Matrix Portal, Wi-Fi, NTP, or OpenWeatherMap. The notebook mirrors the scripts' artwork in a small renderer; it does **not** execute CircuitPython hardware code, and its compact 3×5 font is an approximation of `terminalio.FONT`.

Run the code cells from top to bottom. Change `DISPLAY` to `"base"`, `"halloween"`, `"birthday"`, or `"custom"`. The preview renders an enlarged pixel grid, with play/pause, step, seek, grid, and playback-speed controls. Base and Halloween frames simulate one 30-second display update each; birthday frames step through the scrolling message. Playback speed changes how quickly the simulated frames play.

For your own display, choose `"custom"` and edit `draw_custom(display, frame_index)` in the settings cell. This uses the documented `PreviewDisplay` drawing API below; it does not run an arbitrary CircuitPython script unchanged. Hardware orientation, brightness, and panel color behavior can differ from this offline preview.

In [ ]:
from IPython.display import HTML, display as show_html
import json
import uuid

from matrix_simulator import (
    CUSTOM_PALETTE,
    PreviewDisplay,
    render_frames,
)

## Choose what to preview

For base/Halloween, `FRAME_COUNT` sets how many simulated 30-second updates to generate. The clock starts at `START_TIME`; sample temperatures cycle through `TEMPERATURES` without network requests. For birthday, the frame count controls how much of the scroll is shown. `SPEED` is a playback multiplier (base/Halloween default to 30× so each simulated 30-second update takes about one second; birthday defaults to real animation speed).

In [ ]:
DISPLAY = "base"  # "base", "halloween", "birthday", or "custom"
FRAME_COUNT = 120
START_TIME = "17:25"  # 24-hour HH:MM; used by base and Halloween
TEMPERATURES = [56, 57, 58, 57]  # Sample Fahrenheit values; no API calls
SPEED = 30  # Initial playback multiplier (birthday uses 1)


def draw_custom(display, frame_index):
    """Edit this function to preview a custom 32x32 design.

    Available methods:
      display.clear()
      display.pixel(x, y, color_index)  # clipped to the panel
      display.sprite(rows, x, y)        # rows of palette-index digits
      display.text(text, x, y, color=1, align="left")  # compact 3x5 font

    CUSTOM_PALETTE index 0 is black/transparent; indices 1-7 are colors.
    Use frame_index to animate, for example x = frame_index % 32.
    """
    display.text("CUSTOM", 1, 2, color=5)
    display.sprite(["02220", "22222", "22222", "02220"], frame_index % 27, 15)


if DISPLAY not in {"base", "halloween", "birthday", "custom"}:
    raise ValueError("DISPLAY must be base, halloween, birthday, or custom.")
if DISPLAY == "birthday" and SPEED == 30:
    SPEED = 1

## Render and play

The custom drawing API uses palette indices. `pixel()` safely clips positions outside the 32×32 area; invalid palette indices raise an error instead of silently choosing a color.

In [ ]:
preview = render_frames(
    DISPLAY,
    FRAME_COUNT,
    start_time=START_TIME,
    temperatures=TEMPERATURES,
    custom_draw=draw_custom if DISPLAY == "custom" else None,
)

assert len(preview["frames"]) == FRAME_COUNT
assert all(len(frame) == 32 * 32 for frame in preview["frames"])
assert all(0 <= px < len(preview["palette"]) for frame in preview["frames"] for px in frame)
print("Validated {} frames of 32×32 pixels for {}.".format(FRAME_COUNT, DISPLAY))

payload = json.dumps({
    "scene": preview["scene"],
    "palette": preview["palette"],
    "frames": preview["frames"],
    "notes": preview["notes"],
})
widget_id = "matrix_" + uuid.uuid4().hex
default_speed = SPEED if SPEED in (1, 5, 10, 30, 60, 120) else 30

template = r'''<div id="WIDGET_ID" style="font-family: sans-serif; max-width: 700px;">
  <div style="display:flex; gap:8px; align-items:center; flex-wrap:wrap; margin-bottom:8px;">
    <button data-action="toggle">Play</button>
    <button data-action="prev">◀ Step</button>
    <button data-action="next">Step ▶</button>
    <label>Speed <select data-role="speed">
      <option value="1">1×</option><option value="5">5×</option>
      <option value="10">10×</option><option value="30">30×</option>
      <option value="60">60×</option><option value="120">120×</option>
    </select></label>
    <label><input data-role="grid" type="checkbox" checked> Grid</label>
  </div>
  <canvas data-role="canvas" width="640" height="640" style="width:min(80vw,640px);height:auto;image-rendering:pixelated;border:1px solid #555;"></canvas>
  <div><input data-role="seek" type="range" min="0" max="0" value="0" style="width:min(80vw,640px);">
  <div data-role="status" style="font-family:monospace;"></div></div>
  <script>
  (function() {
    const root = document.getElementById("WIDGET_ID");
    const data = PAYLOAD;
    const canvas = root.querySelector('[data-role="canvas"]');
    const ctx = canvas.getContext("2d");
    const seek = root.querySelector('[data-role="seek"]');
    const status = root.querySelector('[data-role="status"]');
    const speed = root.querySelector('[data-role="speed"]');
    const grid = root.querySelector('[data-role="grid"]');
    const frames = data.frames;
    const colors = data.palette;
    const cell = canvas.width / 32;
    let index = 0;
    let timer = null;
    seek.max = String(frames.length - 1);
    speed.value = "DEFAULT_SPEED";
    if (data.scene === "birthday") speed.value = "1";

    function draw() {
      const frame = frames[index];
      for (let y = 0; y < 32; y++) {
        for (let x = 0; x < 32; x++) {
          ctx.fillStyle = colors[frame[y * 32 + x]];
          ctx.fillRect(x * cell, y * cell, cell, cell);
        }
      }
      if (grid.checked) {
        ctx.beginPath();
        ctx.strokeStyle = "rgba(128,128,128,0.5)";
        ctx.lineWidth = 1;
        for (let n = 0; n <= 32; n++) {
          const p = n * cell + 0.5;
          ctx.moveTo(p, 0); ctx.lineTo(p, canvas.height);
          ctx.moveTo(0, p); ctx.lineTo(canvas.width, p);
        }
        ctx.stroke();
      }
      seek.value = String(index);
      status.textContent = data.scene + " — frame " + (index + 1) + "/" + frames.length + " — " + data.notes[index];
    }
    function stop() {
      if (timer !== null) clearInterval(timer);
      timer = null;
      root.querySelector('[data-action="toggle"]').textContent = "Play";
    }
    function step(amount) {
      index = (index + amount + frames.length) % frames.length;
      draw();
    }
    function play() {
      stop();
      const multiplier = Number(speed.value);
      const ms = data.scene === "birthday" ? 60 / multiplier : 30000 / multiplier;
      timer = setInterval(function() { step(1); }, Math.max(16, ms));
      root.querySelector('[data-action="toggle"]').textContent = "Pause";
    }
    root.querySelector('[data-action="toggle"]').addEventListener("click", function() { timer === null ? play() : stop(); });
    root.querySelector('[data-action="prev"]').addEventListener("click", function() { stop(); step(-1); });
    root.querySelector('[data-action="next"]').addEventListener("click", function() { stop(); step(1); });
    seek.addEventListener("input", function() { index = Number(seek.value); draw(); });
    speed.addEventListener("change", function() { if (timer !== null) play(); });
    grid.addEventListener("change", draw);
    draw();
  })();
  </script>
</div>'''

html = (
    template.replace("WIDGET_ID", widget_id)
    .replace("PAYLOAD", payload)
    .replace("DEFAULT_SPEED", str(default_speed))
)
show_html(HTML(html))